In [118]:
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F

from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader

from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms.functional as VF
from PIL import Image
import time

device = 'cuda'
print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.8.0+cu128


In [41]:
transform = transforms.Compose([
    transforms.ToTensor()
])

In [42]:
train_dataset = datasets.MNIST(
    root="./DATA",
    train=True,
    download=True,
    transform=transform)

test_dataset = datasets.MNIST(
    root="./DATA",
    train=False,
    download=True,
    transform=transform)    

In [5]:
# tensorboard --logdir runs --bind_all --load_fast=false --samples_per_plugin images=100
writer = SummaryWriter('runs/exp')
for j in range(10):
    for i in range(0, 90, 5):
        image, label = train_dataset[j]
        image = VF.rotate(image, i)
        fig = plt.figure(figsize=(16, 4))
        plt.imshow(image, cmap="gray")
        plt.title(f"Label: {label}")
        writer.add_figure(f'Rotate_{j}', fig, global_step=i)
        plt.close(fig)
writer.close()        

In [105]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 1024)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(1024, 256)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)        
        return x

In [140]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=512,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=512,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    drop_last=False)

model = SimpleNN().to(device)
optimizer = optim.Adam(
    params=model.parameters(),
    lr=0.01,
    weight_decay=0.0001
)
criterion = nn.CrossEntropyLoss()

writer = SummaryWriter(f'runs/model_{int(time.time())}')
global_step = 0
for epoch in range(100):
    running_loss = 0.0
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()


        total_params_norm = 0.0
        total_params_count = 0.0
        total_grad_norm = 0.0
        for name, param in model.named_parameters():
            if param.requires_grad:
                param_norm = param.data.norm(2)
                param_norm_scaled = param_norm / (param.numel() ** 0.5)
                total_params_norm += param_norm.item() ** 2
                total_params_count += param.numel()
                writer.add_scalar(f"Weights/{name}_norm_scaled", param_norm_scaled, global_step)

                grad_norm = param.grad.data.norm(2)
                total_grad_norm += grad_norm.item() ** 2
                writer.add_scalar(f"Gradient/{name}_norm", grad_norm, global_step)

                grad_ratio = grad_norm / param_norm
                writer.add_scalar(f"Gradient_ratio/{name}", grad_ratio, global_step)
        total_params_norm = (total_params_norm / total_params_count) ** 0.5
        writer.add_scalar(f"Weights/total_norm", total_params_norm, global_step)
        total_grad_norm = total_grad_norm ** 0.5
        writer.add_scalar("Gradients/total_norm_preclip", total_grad_norm, global_step)
        
        clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()        
        
        writer.add_scalar("Loss/train_batch", loss.item(), global_step)
        global_step += 1
        running_loss += loss.item()
    epoch_loss = running_loss / len(train_loader)
    writer.add_scalar("Loss/train_epoch", epoch_loss, epoch)

    model.eval()
    running_loss_test = 0.0
    n_obs_test = 0
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = criterion(output, target)
        running_loss_test += loss.item() * len(data)
        n_obs_test += len(data)
    epoch_loss_test = running_loss_test / n_obs_test
    writer.add_scalar("Loss/test_epoch", epoch_loss_test, epoch)
    
writer.close()

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7339ea9be840>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7339ea9be840>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16